## Fase 1: Data Engineering

Abran el archivo .json usando la librería json de Python.
Extraigan los datos para construir una única tabla maestra (DataFrame) con 3 columnas: Origen, Destino y Relacion.

Reto A (La Capa Social): Extraigan los "retweets". Si Usuario_A retuitea a Usuario_B, eso es una relación tipo "Retweet". Ignoren auto-retweets. (Ojo con las minúsculas/mayúsculas).

Reto B (La Capa de Infraestructura): En el JSON, la IP y la Tarjeta son metadatos individuales. Deberán ingeniárselas (recomendación: extraigan los metadatos a un DataFrame y hagan un merge del dataframe consigo mismo) para conectar a los usuarios. Si Usuario_A y Usuario_B tienen la misma Tarjeta, agreguen una fila a su tabla con relación tipo "Misma_Tarjeta".

In [ ]:
import json
import pandas as pd

with open('api_troll_farm.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)

display(df.head())


,account_id,metadata,activity
0,TROLL_BOT_1,"{'last_ip': '104.28.19.112', 'payment_hash': '...","{'retweets': ['real_user_11', 'TROLL_BOT_1', '..."
1,real_user_14,"{'last_ip': '189.168.1.196', 'payment_hash': '...","{'retweets': ['real_user_15', 'real_user_11']}"
2,real_user_13,"{'last_ip': '189.168.1.184', 'payment_hash': N...","{'retweets': ['real_user_3', 'real_user_9']}"
3,TROLL_BOT_5,"{'last_ip': '104.28.19.112', 'payment_hash': '...","{'retweets': ['real_user_12', 'TROLL_BOT_1', '..."
4,real_user_12,"{'last_ip': '189.168.1.51', 'payment_hash': 'M...","{'retweets': ['real_user_12', 'real_user_15']}"


In [ ]:
# RETO A
import pandas as pd

retweets_list = []

for record in data:
    usuario_a = record.get('user', {}).get('screen_name', '').strip()

    retweeted_status = record.get('retweeted_status')
    if retweeted_status and isinstance(retweeted_status, dict):
        usuario_b = retweeted_status.get('user', {}).get('screen_name', '').strip()

        if usuario_a.lower() != usuario_b.lower() and usuario_a and usuario_b:
            retweets_list.append({
                'Origen': usuario_a,
                'Destino': usuario_b,
                'Relacion': 'Retweet'
            })

df_master = pd.DataFrame(retweets_list)

print(f"{len(df_master)} retweets")
display(df_master.head())

0 retweets


""


In [ ]:
# RETO B
import pandas as pd

infra_list = []
for record in data:
    user = record.get('account_id')
    metadata = record.get('metadata', {})
    if metadata:
        payment_hash = metadata.get('payment_hash')
        if payment_hash:
            infra_list.append({
                'Usuario': user,
                'Tarjeta': payment_hash
            })

df_infra = pd.DataFrame(infra_list)

df_merged = pd.merge(df_infra, df_infra, on='Tarjeta', suffixes=('_A', '_B'))

df_same_card = df_merged[df_merged['Usuario_A'] != df_merged['Usuario_B']].copy()

infra_relations = pd.DataFrame({
    'Origen': df_same_card['Usuario_A'],
    'Destino': df_same_card['Usuario_B'],
    'Relacion': 'Misma_Tarjeta'
})

df_master = pd.concat([df_master, infra_relations], ignore_index=True)

print(f"{len(infra_relations)} relaciones tipo 'Misma_Tarjeta'")
print(f"Total de filas en df_master: {len(df_master)}")
display(df_master.tail())


42 relaciones tipo 'Misma_Tarjeta'
Total de filas en df_master: 42


,Origen,Destino,Relacion
37,TROLL_BOT_2,TROLL_BOT_5,Misma_Tarjeta
38,TROLL_BOT_2,troll_bot_1,Misma_Tarjeta
39,TROLL_BOT_2,TROLL_BOT_4,Misma_Tarjeta
40,TROLL_BOT_2,TROLL_BOT_6,Misma_Tarjeta
41,TROLL_BOT_2,TROLL_BOT_3,Misma_Tarjeta


## Fase 2: El Grafo Multicapa

Pasen su tabla maestra purificada a NetworkX utilizando nx.MultiDiGraph().
Comprueben que su grafo cargó correctamente imprimiendo el número total de Nodos y Aristas.

In [ ]:
import networkx as nx

B_G = nx.MultiDiGraph()

for _, row in df_master.iterrows():
    B_G.add_edge(
        row['Origen'],
        row['Destino'],
        relacion=row['Relacion']
    )

print(f"Nodos totales: {B_G.number_of_nodes()}")
print(f"Aristas totales: {B_G.number_of_edges()}")

Nodos totales: 7
Aristas totales: 42


## Fase 3: El Slicing
Los trolls son lo suficientemente listos para esconderse en el ruido de los retweets, pero cometieron un error de novatos: todos pagaron su cuenta bot con la misma tarjeta de crédito robada.

Filtren su MultiGrafo para extraer exclusivamente las aristas cuya Relación sea "Misma_Tarjeta".
Metan esas aristas en un grafo no dirigido simple (nx.Graph()).
Usen la función nx.connected_components() sobre este nuevo sub-grafo para encontrar a la tribu coludida financieramente.

In [ ]:
# grafo simple no dirigido
card_graph = nx.Graph()

for u, v, key, data in B_G.edges(keys=True, data=True):
    if data.get('relacion') == 'Misma_Tarjeta':
        card_graph.add_edge(u, v)

tribus = list(nx.connected_components(card_graph))

tribu_fraude = 0

print(f"{len(tribus)} tribus financieras.")
for idx, tribu in enumerate(tribus, 1):
    print(f"Tribu #{idx} (tamaño: {len(tribu)}): {tribu}")
    tribu_fraude = tribu



1 tribus financieras.
Tribu #1 (tamaño: 7): {'TROLL_BOT_5', 'TROLL_BOT_1', 'TROLL_BOT_4', 'TROLL_BOT_3', 'TROLL_BOT_2', 'TROLL_BOT_6', 'troll_bot_1'}


In [ ]:
print(df_infra[df_infra['Usuario'].isin(tribu_fraude)]['Tarjeta'].unique()[0])

VISA-4455-XXXX-9090


### El Veredicto: Respondan en una celda al final del notebook:

¿Cuáles son los nombres exactos (account_id) de los bots que conforman el anillo de fraude?  
{'TROLL_BOT_5', 'TROLL_BOT_1', 'TROLL_BOT_4', 'TROLL_BOT_3', 'TROLL_BOT_2', 'TROLL_BOT_6', 'troll_bot_1'}

¿Cuál es el Hash de la tarjeta de crédito comprometida que el equipo de soporte técnico debe bloquear ahora mismo?   
VISA-4455-XXXX-9090

Si tuvieran que entrenar un modelo de predicción, ¿cómo le pasarían este descubrimiento al modelo tabular (La matriz X de entrenamiento)?   
Crear una nueva columna binaria o usar metricas de grafo como el grado de centralidad del subgrafo de tarjetas.
